<a href="https://colab.research.google.com/github/yousufcoxs/Chat_Application/blob/master/Knnfromscratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from collections import Counter

class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X_train, y_train):
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)

    def predict(self, X_test):
        predictions = []
        for x in X_test:
            distances = [np.linalg.norm(x - x_train) for x_train in self.X_train]
            k_indices = np.argsort(distances)[:self.k]
            k_nearest_labels = [self.y_train[i] for i in k_indices]
            most_common = Counter(k_nearest_labels).most_common(1)[0][0]
            predictions.append(most_common)
        return np.array(predictions)


In [ ]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

def confusion_matrix(y_true, y_pred, labels):
    matrix = np.zeros((len(labels), len(labels)), dtype=int)
    label_to_index = {label: i for i, label in enumerate(labels)}
    for t, p in zip(y_true, y_pred):
        matrix[label_to_index[t]][label_to_index[p]] += 1
    return matrix

def precision_recall_f1(y_true, y_pred, labels):
    matrix = confusion_matrix(y_true, y_pred, labels)
    precisions, recalls, f1s = [], [], []
    for i in range(len(labels)):
        TP = matrix[i][i]
        FP = np.sum(matrix[:, i]) - TP
        FN = np.sum(matrix[i, :]) - TP
        precision = TP / (TP + FP) if (TP + FP) != 0 else 0
        recall = TP / (TP + FN) if (TP + FN) != 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return precisions, recalls, f1s


In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

data = load_iris()
X, y = data.data, data.target
labels = list(set(y))


In [ ]:
best_accuracy = 0
best_k = 0
best_split = 0

for k in range(1, 11):
    for test_size in [0.2, 0.3, 0.4]:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
        knn = KNN(k=k)
        knn.fit(X_train, y_train)
        preds = knn.predict(X_test)
        acc = accuracy(y_test, preds)
        if acc > best_accuracy:
            best_accuracy = acc
            best_k = k
            best_split = test_size


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=best_split, random_state=42)
final_knn = KNN(k=best_k)
final_knn.fit(X_train, y_train)
final_preds = final_knn.predict(X_test)

final_accuracy = accuracy(y_test, final_preds)
final_confusion_matrix = confusion_matrix(y_test, final_preds, labels)
final_precisions, final_recalls, final_f1s = precision_recall_f1(y_test, final_preds, labels)

print(f"Best k: {best_k}")
print(f"Best test size: {best_split}")
print(f"Final Accuracy: {final_accuracy}")
print("Confusion Matrix:")
display(final_confusion_matrix)
print("Precision, Recall, F1-score for each class:")
for i, label in enumerate(labels):
    print(f"Class {label}: Precision={final_precisions[i]:.2f}, Recall={final_recalls[i]:.2f}, F1-score={final_f1s[i]:.2f}")

Best k: 1
Best test size: 0.2
Final Accuracy: 1.0
Confusion Matrix:


array([[10,  0,  0],
       [ 0,  9,  0],
       [ 0,  0, 11]])

Precision, Recall, F1-score for each class:
Class 0: Precision=1.00, Recall=1.00, F1-score=1.00
Class 1: Precision=1.00, Recall=1.00, F1-score=1.00
Class 2: Precision=1.00, Recall=1.00, F1-score=1.00


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts = ["News about sports", "Politics and economy", "Match updates", "placeholder text 1", "placeholder text 2"]
labels = ["sports", "politics", "sports", "placeholder_label_1", "placeholder_label_2"]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts).toarray()

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

model = KNeighborsClassifier(n_neighbors=best_k)
model.fit(X_train, y_train)
sk_preds = model.predict(X_test)

labels = [0, 1, 2]

print("Accuracy:", accuracy(y_test, sk_preds))
print("Confusion Matrix:", confusion_matrix(y_test, sk_preds, labels))
print("Precision, Recall, F1:", precision_recall_f1(y_test, sk_preds, labels))

Accuracy: 1.0
Confusion Matrix: [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
Precision, Recall, F1: ([np.float64(1.0), np.float64(1.0), np.float64(1.0)], [np.float64(1.0), np.float64(1.0), np.float64(1.0)], [np.float64(1.0), np.float64(1.0), np.float64(1.0)])
